[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/feibertciin/RN/blob/main/CNN-tutor.ipynb)

# 🧠 Tema 2: Estructura y Funcionamiento de las Redes Neuronales Recurrentes (RNN)

## 📘 Introducción al Cuaderno

Este cuaderno ha sido diseñado como un **material didáctico profundo** para comprender la arquitectura, el funcionamiento y la implementación de las Redes Neuronales Recurrentes (RNN) y sus variantes especializadas: **LSTM** (Long Short-Term Memory) y **GRU** (Gated Recurrent Unit).

A diferencia de las redes feed-forward (como las MLP o CNN), las RNN están diseñadas específicamente para el **procesamiento de datos secuenciales** (texto, series temporales, audio), ya que poseen una "memoria" interna que les permite conservar el contexto de las entradas anteriores.

---

## 🎯 1. Objetivos del Tutorial

1. **Comprender** la arquitectura básica de una RNN simple y su mecanismo de estado oculto.
2. **Identificar** las limitaciones de las RNN simples (problemas de gradiente de desaparición/explosión).
3. **Analizar** la arquitectura de las LSTM y el funcionamiento de sus tres puertas (entrada, olvido, salida).
4. **Explorar** las GRU como una alternativa simplificada y computacionalmente eficiente a las LSTM.
5. **Implementar** ejemplos prácticos en Python usando TensorFlow/Keras y NumPy (desde cero).

## 🔄 2. Concepto Fundamental: ¿Qué es una RNN?

Las Redes Neuronales Recurrentes (RNN) están diseñadas para el manejo de **datos secuenciales**. Su característica distintiva es que poseen **bucles** (conexiones recurrentes) que permiten a la red recordar información de entradas anteriores.

### 🔑 Mecanismo Clave: El Estado Oculto ($h_t$)
A diferencia de las redes de propagación hacia adelante, en una RNN los datos fluyen de un paso de la secuencia al siguiente, preservando el contexto a lo largo del tiempo. En cada paso de tiempo $t$:

1. La red recibe la entrada actual ($x_t$).
2. Recibe el estado oculto del paso anterior ($h_{t-1}$).
3. Actualiza continuamente un **estado interno oculto** ($h_t$) que acumula y transmite información relevante.

> 💡 **Aplicaciones típicas**: Reconocimiento de voz (Siri, Alexa), texto predictivo, autocompletado, generación de música y reconocimiento de escritura a mano.

## 🛠️ 3. Implementación de una RNN Simple con TensorFlow/Keras

A continuación, construiremos una RNN simple para predecir el siguiente número en una secuencia (en este caso, la suma de los elementos de la secuencia).

> ⚠️ **Nota sobre buenas prácticas en Keras**: El texto original menciona una advertencia (`UserWarning`) al pasar `input_shape` directamente a la capa `SimpleRNN`. La práctica moderna y recomendada es usar una capa `Input` explícita como primera capa del modelo `Sequential`. El código a continuación aplica esta mejora para evitar la advertencia.

In [ ]:
# ============================================================
# EJEMPLO 1: Red Neuronal Recurrente (RNN) Simple
# ============================================================
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Input

# 1. Generar datos ficticios para demostración
def generate_data(seq_length, num_samples):
    # Forma de entrada: (num_samples, seq_length, features)
    X = np.random.rand(num_samples, seq_length, 1)
    # Objetivo: suma de la secuencia
    y = np.array([np.sum(x) for x in X])
    return X, y

# Parámetros
seq_length = 10
num_samples = 1000

# Generar datos
X, y = generate_data(seq_length, num_samples)
print(f"Forma de X: {X.shape} | Forma de y: {y.shape}")

# 2. Construir el modelo RNN
model_rnn = Sequential([
    # Buena práctica: Usar Input explícito en lugar de input_shape en la capa
    Input(shape=(seq_length, 1)),
    # Capa RNN con 50 unidades y activación tanh
    SimpleRNN(50, activation='tanh'),
    # Capa de salida densa para la predicción final (regresión)
    Dense(1)
])

# 3. Compilar el modelo
model_rnn.compile(optimizer='adam', loss='mean_squared_error')

# 4. Entrenar el modelo
print("\n🚀 Iniciando entrenamiento...")
history = model_rnn.fit(X, y, epochs=20, batch_size=32, verbose=1)

# 5. Ejemplo de predicción
test_data = np.random.rand(1, seq_length, 1)
predicted_sum = model_rnn.predict(test_data, verbose=0)
real_sum = np.sum(test_data)

print(f"\n✅ Suma real de la secuencia: {real_sum:.4f}")
print(f"✅ Suma predicha por la RNN:  {predicted_sum[0][0]:.4f}")

### ⚠️ Limitaciones de las RNN Simples

A pesar de su elegancia conceptual, las RNN sencillas suelen tener dificultades para retener información de intervalos de tiempo anteriores. Esto se debe a dos problemas fundamentales durante el retropropagación del error a través del tiempo (BPTT):

1. **Desvanecimiento del gradiente (Vanishing Gradient)**: Los gradientes se vuelven exponencialmente pequeños, impidiendo que la red aprenda dependencias a largo plazo.
2. **Explosión del gradiente (Exploding Gradient)**: Los gradientes crecen exponencialmente, causando inestabilidad numérica.

Para solucionar esto, se desarrollaron arquitecturas especializadas: **LSTM** y **GRU**.

## 🧠 4. Redes de Memoria a Corto y Largo Plazo (LSTM)

Las **LSTM** (Long Short-Term Memory) son un tipo especial de RNN diseñadas para aprender y recordar información en secuencias largas. Incorporan mecanismos que regulan el flujo de información, permitiéndoles recordar datos relevantes durante periodos mucho más largos.

### 🚪 Arquitectura de la Celda LSTM
Cada unidad LSTM contiene una **celda de memoria** y **tres puertas** principales:

| Puerta | Función |
|---|---|
| **Puerta de Olvido (Forget Gate)** | Decide qué información debe descartarse del estado de la celda anterior. |
| **Puerta de Entrada (Input Gate)** | Decide cuánta información nueva de la entrada actual debe almacenarse en la celda. |
| **Puerta de Salida (Output Gate)** | Controla la cantidad de información de la celda de memoria que debe transferirse al siguiente estado oculto. |

> 💡 **Aplicaciones**: Modelado lingüístico, reconocimiento de voz, predicción de series temporales y traducción automática.

In [ ]:
# ============================================================
# EJEMPLO 2: Red Neuronal LSTM con TensorFlow/Keras
# ============================================================
from tensorflow.keras.layers import LSTM

# 1. Definir dimensiones de entrada
timesteps = 10   # Número de pasos temporales (longitud de la secuencia)
features = 5     # Número de características por paso temporal

# 2. Definir el modelo
model_lstm = Sequential([
    Input(shape=(timesteps, features)),
    
    # Capa LSTM con configuraciones personalizadas
    LSTM(
        units=50,                      # Número de unidades LSTM (dimensión del estado oculto)
        activation='tanh',             # Activación del estado de la celda
        recurrent_activation='sigmoid',# Activación de las puertas (sigmoid limita la salida entre 0 y 1)
        return_sequences=False         # False: solo devuelve la salida del último paso temporal
    ),
    
    # Capa densa para la salida (ejemplo: clasificación binaria)
    Dense(units=1, activation='sigmoid')
])

# 3. Compilar el modelo
model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Resumen del modelo
print("📊 Resumen de la arquitectura LSTM:")
model_lstm.summary()

### 📌 Nota sobre `return_sequences`
Si el modelo incluye **varias capas LSTM apiladas**, es necesario configurar `return_sequences=True` para cada capa LSTM **excepto la última**. Esto garantiza que cada capa proporcione su salida de secuencia completa (todos los pasos de tiempo) a la siguiente capa de la pila, en lugar de solo el último paso.

## ⚡ 5. Unidades Recurrentes Compuertadas (GRU)

Las **GRU** (Gated Recurrent Unit) son un tipo específico de arquitectura RNN, **más sencilla que las LSTM**, que simplifican los mecanismos internos necesarios para la gestión de la memoria.

### 🚪 Arquitectura de la Celda GRU
A diferencia de las LSTM, las GRU fusionan el estado de la celda y el estado oculto en un único vector, y utilizan solo **dos puertas**:

| Puerta | Función |
|---|---|
| **Puerta de Actualización (Update Gate, $z_t$)** | Determina el grado en que se mantiene el estado oculto anterior frente a su actualización con nueva información. |
| **Puerta de Reinicio (Reset Gate, $r_t$)** | Controla cuánta información anterior debe olvidarse antes de calcular el estado candidato. |

> 💡 **Ventajas**: Tiempos de entrenamiento más rápidos, menor complejidad computacional y, a menudo, un rendimiento competitivo frente a las LSTM en conjuntos de datos más pequeños.

In [ ]:
# ============================================================
# EJEMPLO 3: Celda GRU construida desde cero con NumPy
# ============================================================

class GRUCell:
    def __init__(self, tamaño_entrada, tamaño_oculto):
        self.tamaño_entrada = tamaño_entrada
        self.tamaño_oculto = tamaño_oculto

        # Inicialización de matrices de pesos (aleatoria para demostración)
        self.Wz = np.random.randn(tamaño_oculto, tamaño_entrada)   # Pesos de compuerta de actualización (entrada)
        self.Uz = np.random.randn(tamaño_oculto, tamaño_oculto)   # Pesos recurrentes de actualización (oculto)
        self.Wr = np.random.randn(tamaño_oculto, tamaño_entrada)  # Pesos de compuerta de reinicio (entrada)
        self.Ur = np.random.randn(tamaño_oculto, tamaño_oculto)   # Pesos recurrentes de reinicio (oculto)
        self.Wh = np.random.randn(tamaño_oculto, tamaño_entrada)  # Pesos de estado oculto candidato (entrada)
        self.Uh = np.random.randn(tamaño_oculto, tamaño_oculto)   # Pesos recurrentes del estado oculto candidato

    def sigmoid(self, x):
        # Función de activación sigmoide (limita valores entre 0 y 1, ideal para puertas)
        return 1 / (1 + np.exp(-x))

    def tanh(self, x):
        # Función de activación tangente hiperbólica (limita valores entre -1 y 1)
        return np.tanh(x)

    def forward(self, x_t, h_prev):
        """
        Pase hacia adelante de la celda GRU.
        x_t: Entrada en el tiempo t
        h_prev: Estado oculto en el tiempo t-1
        """
        # 1. Compuerta de actualización (z_t)
        z_t = self.sigmoid(np.dot(self.Wz, x_t) + np.dot(self.Uz, h_prev))

        # 2. Compuerta de reinicio (r_t)
        r_t = self.sigmoid(np.dot(self.Wr, x_t) + np.dot(self.Ur, h_prev))

        # 3. Estado oculto candidato (h_tilde)
        # Se aplica la puerta de reinicio al estado oculto anterior
        h_tilde = self.tanh(np.dot(self.Wh, x_t) + np.dot(self.Uh, r_t * h_prev))

        # 4. Nuevo estado oculto (h_t)
        # Combinación lineal del estado anterior y el candidato, ponderada por z_t
        h_t = (1 - z_t) * h_prev + z_t * h_tilde

        return h_t, z_t, r_t

# --- Ejemplo de uso ---
tamaño_entrada = 3
tamaño_oculto = 2
gru_cell = GRUCell(tamaño_entrada, tamaño_oculto)

# Entrada de muestra y estado oculto previo
x_t = np.random.randn(tamaño_entrada)
h_prev = np.random.randn(tamaño_oculto)

# Pase hacia adelante
h_t, z_t, r_t = gru_cell.forward(x_t, h_prev)

print("🔍 Resultados del paso forward de la celda GRU:")
print(f"Nuevo estado oculto (h_t):       {np.round(h_t, 4)}")
print(f"Compuerta de actualización (z_t): {np.round(z_t, 4)}")
print(f"Compuerta de reinicio (r_t):      {np.round(r_t, 4)}")

## ⚖️ 6. Resumen Comparativo: RNN vs LSTM vs GRU

Las redes neuronales recurrentes constituyen la base para el modelado de datos secuenciales, pero sus variantes responden a necesidades diferentes:

| Característica | RNN Simple | LSTM | GRU |
|---|---|---|---|
| **Mecanismo de memoria** | Estado oculto único | Celda de memoria + Estado oculto | Estado oculto único (fusionado) |
| **Número de puertas** | 0 (solo activación) | 3 (Entrada, Olvido, Salida) | 2 (Actualización, Reinicio) |
| **Dependencias a largo plazo** | Muy limitadas (Vanishing Gradient) | Excelentes | Muy buenas |
| **Complejidad computacional** | Baja | Alta | Media |
| **Velocidad de entrenamiento** | Rápida | Lenta | Intermedia/Rápida |
| **Mejor uso** | Secuencias muy cortas | Secuencias largas y complejas | Secuencias medianas, recursos limitados |

> 🎯 **Conclusión clave**: La elección entre estos tres tipos de redes depende del equilibrio requerido entre **capacidad de memoria**, **eficiencia computacional** y **complejidad del problema** a resolver.

## 📖 7. Glosario de Términos

| Término | Definición |
|---|---|
| **RNN** | Red Neuronal Recurrente. Arquitectura con bucles para procesar datos secuenciales. |
| **Estado Oculto ($h_t$)** | Vector que resume la información de los pasos anteriores de la secuencia. |
| **BPTT** | Backpropagation Through Time. Algoritmo de entrenamiento para RNNs que propaga el error a través de los pasos de tiempo. |
| **Vanishing Gradient** | Problema donde los gradientes se vuelven cercanos a cero, impidiendo el aprendizaje de dependencias lejanas. |
| **LSTM** | Long Short-Term Memory. RNN especializada con celda de memoria y 3 puertas para retener información a largo plazo. |
| **GRU** | Gated Recurrent Unit. Variante simplificada de LSTM con 2 puertas y menor costo computacional. |
| **Puerta (Gate)** | Mecanismo (usualmente con activación sigmoide) que decide cuánta información dejar pasar (valor entre 0 y 1). |
| **`return_sequences`** | Parámetro en Keras que, si es `True`, devuelve la salida de la RNN para *todos* los pasos de tiempo, no solo el último. |
| **Celda Candidata ($\tilde{h}_t$)** | En GRU/LSTM, es el nuevo contenido de memoria propuesto antes de ser filtrado por las puertas. |

## 🎓 8. Conclusiones y Próximos Pasos

### 🔹 Conclusiones del Tema
1. Las **RNN simples** son el punto de partida conceptual para entender el procesamiento secuencial, pero su utilidad práctica es limitada debido al desvanecimiento del gradiente.
2. Las **LSTM** resuelven este problema mediante un sofisticado sistema de tres puertas, siendo el estándar de oro para secuencias largas y complejas (como traducción automática).
3. Las **GRU** ofrecen un equilibrio excelente: simplifican la arquitectura de la LSTM (fusionando estados y eliminando una puerta), logrando un rendimiento muy similar con un entrenamiento más rápido y menos parámetros.

### 🔹 Recomendaciones para el Aprendizaje
Para dominar estas arquitecturas, se recomienda seguir esta ruta:

1. **Fundamentos**: Comprender bien las estructuras básicas y sus limitaciones matemáticas (como se mostró en la celda de NumPy).
2. **Frameworks**: Explorar marcos de alto nivel como **TensorFlow/Keras** o **PyTorch** para agilizar la construcción de modelos.
3. **Experimentación Práctica**: Aplicar estos modelos a conjuntos de datos reales:
   - *Series temporales*: Predicción de precios de acciones o temperatura.
   - *Procesamiento de Lenguaje Natural (NLP)*: Clasificación de texto o generación de secuencias.
4. **Optimización**: Aprender a ajustar hiperparámetros como el número de unidades, el uso de `Dropout` recurrente y la tasa de aprendizaje para evitar el sobreajuste.

---

### 🙌 ¡Felicitaciones!
Has completado el estudio teórico y práctico de las arquitecturas recurrentes. Este cuaderno te proporciona la base sólida necesaria para avanzar hacia modelos secuenciales más complejos o incluso hacia arquitecturas de vanguardia como los **Transformers**.